# Load the questions

In [ ]:
import json
from dataclasses import dataclass
from typing import TypeAlias
import uuid
import re
from tqdm.notebook import tqdm
from langchain.messages import HumanMessage

from model_0 import agent as agent_0
from model_1 import agent as agent_1

with open("benchmark.json") as f:
    qa = json.load(f)


E0000 00:00:1778466257.421163 1003104 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1778466257.421260 1003104 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1778466257.421266 1003104 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1778466257.421267 1003104 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1778466257.421269 1003104 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the def

Found 0 new videos to add to collection youtube_videos_test_text.


Building KB: 0it [00:00, ?it/s]


Found 0 new videos to add to collection youtube_videos_test_hybrid.


Building KB: 0it [00:00, ?it/s]


# Load the architectures to evaluate
| Ablation              | Short-Term Memory |        KB        | Vector DB Indexing | Bias Analysis Sub-Agent |
|-----------------------|:-----------------:|:----------------:|:------------------:|:-----------------------:|
| 0. **Vanilla**        |         Y         |       N/A        |        N/A         |            N            |
| 1. **Text RAG**       |         Y         |       CSV        |        Text        |            N            |
| 2. **MM-RAG**         |         Y         | CSV + Thumbnails |       Hybrid       |            N            |
| 3. **Agentic MM-RAG** |         Y         | CSV + Thumbnails |       Hybrid       |            Y            |

In [ ]:
user_id = 25
agents = [
    (agent_0, uuid.uuid4().hex),
    (agent_1, uuid.uuid4().hex),
    # (agent_2, uuid.uuid4().hex),
    # (agent_3, uuid.uuid4().hex),
]

for category, pairs in tqdm(qa.items(), desc="Category"):
    prompt = "Retrieve the last 5 watched videos, then answer the following questions as a bulleted list in a concise form:\n"
    questions = "\n".join(f"* {pair["question"]}" for pair in pairs)
    golden_answers = "\n".join(f"* {pair["answer"]}" for pair in pairs)
    prompt += questions
    
    print(f"\nQ:\n{prompt}")
    print(f"\nA:\n{golden_answers}")

    for i, (agent, thread_id) in enumerate(agents):
        result = agent.invoke(
            {"messages": [HumanMessage(content=prompt)]},
            config={"configurable": {"thread_id": thread_id}},
            context={"user_id": user_id},
        )

        answers = result.get("messages", [])[-1].content
        print(f"\nA{i}")
        print(answers)

        for item, answer in zip(pairs, re.findall(r"(?m)^\*\s*(.*)$", answers)):
            item[f"model_{i}"] = answer

Category:   0%|          | 0/1 [00:00<?, ?it/s]


Q:
Retrieve the last 5 watched videos, then answer the following questions as a bulleted list in a concise form:
* What is the exact watch date and time for the video titled 'Why Small Biases Matter Over Time'?
* Which video ID corresponds to the video 'SkiVel'?
* What is the title of the video watched on 2018-09-11 at 15:42:17?
* Which title has the video with ID 8DUSSbJPfwU?

A:
* 2018-09-11 16:59:59
* F8wJdvXK5yU
* Why small biases matter
* The system nudges more than you think

A0
The last 5 watched videos are: 
1. 'Why Small Biases Matter Over Time' (ID: 3rVj9sXZwFw) watched on 2022-01-01 at 10:00:00
2. 'SkiVel' (ID: 5tHJvkRRxRY) watched on 2022-01-02 at 12:00:00
3. 'Understanding Polarization' (ID: 8DUSSbJPfwU) watched on 2018-09-11 at 15:42:17
4. 'The Dangers of Sensationalism' (ID: 2qRJHKvbFvz) watched on 2022-01-03 at 14:00:00
5. 'Clickbait and Its Consequences' (ID: 1aBcDeFgHiJ) watched on 2022-01-04 at 16:00:00

Here are the answers to the questions:
* The exact watch date 

In [ ]:
with open("benchmark.json", "w") as f:
    json.dump(qa, f, indent=2, ensure_ascii=False)